# Step 1 - Explore dataset
Loading the dataset into a table (a DataFrame). And showing the first rows and the column names

In [ ]:
import pandas

df = pandas.read_csv("titanic.csv")
df.head()

### Identify missing values (which columns have gaps?).

In [ ]:
df.info()

Transform data to useful data

In [ ]:
df["pclass"] = df["pclass"].map({"1st":1, "2nd":2, "3rd":3}).astype(int)
df.head()

## Calculating survival percentage

In [ ]:
survival_percentage = df["survived"].mean() * 100
print(f"Survival percentage: {survival_percentage:.2f}%")

## Calculating survival chance per ticket class

In [ ]:
import matplotlib.pyplot as plt

class_survival = df.groupby("pclass")["survived"].mean() * 100
class_survival.plot(kind="bar")

plt.ylabel("Survival %")
plt.title("Survival by Ticket Class")
plt.show()

print(class_survival)

## Calculating survival chance per gender

In [ ]:
gender_survival = df.groupby("sex")["survived"].mean() * 100
gender_survival.plot(kind="bar")

plt.ylabel("Survival %")
plt.xlabel("Gender")
plt.title("Survival by Gender")
plt.show()

print(gender_survival)

## Calculating survival chance per age group

Creating age groups in steps of 10 years

In [ ]:
df["age_group"] = pandas.cut(df["age"], bins=[0,10,20,30,40,50,60,70,80,90,100])

age_survival = df.groupby("age_group")["survived"].mean() * 100
age_survival.plot(kind="bar")

plt.ylabel("Survival %")
plt.title("Survival by Age Group")
plt.show()

print(age_survival)

# Step 2 - Building a first model

In [ ]:
import pandas as pd

x = df[["pclass", "sex", "age"]].copy()
x

In [ ]:
y = df["survived"]
y

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Fill missing ages with median age & Convert "male" "female" to boolean
x["age"] = x["age"].fillna(x["age"].median())
X = pd.get_dummies(x, columns=["sex"], drop_first=True)

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

model = DecisionTreeClassifier()
model.fit(x_train, y_train)

accuracy = model.score(x_test, y_test)
print(f"Model accuracy: {accuracy*100:.2f}%")

## Stupid Baseline 1: Predict Nobody Survives

In [ ]:
from sklearn.metrics import accuracy_score

baseline_pred = [0] * len(y_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print(f"Baseline (nobody survives) accuracy: {baseline_accuracy*100:.2f}%")

## Stupid baseline 2: Predict Everyone Survives

In [ ]:
baseline_pred_all_survive = [1] * len(y_test)
baseline_accuracy_all = accuracy_score(y_test, baseline_pred_all_survive)

print(f"Baseline (everyone survives) accuracy: {baseline_accuracy_all*100:.2f}%")

## Stupid baseline 3: Predict Most Common Class

In [ ]:
most_common_class = y_train.mode()[0]
baseline_pred_majority = [most_common_class] * len(y_test)
baseline_majority_accuracy = accuracy_score(y_test, baseline_pred_majority)

print(f"Baseline (majority class) accuracy: {baseline_majority_accuracy*100:.2f}%")

# Accuracy conclusion:
- Stupid baseline (everyone dies): 63.88%
- Baseline (everyone survives) accuracy: 36.12%
- Baseline (majority class) accuracy: 63.88%


- Model: 81.37%

81.37% − 63.88% = 17.49% improvement


## Model predictions & info

In [ ]:
new_passenger = pd.DataFrame({
    "pclass": [1],
    "age": [39],
    "sex_male": [1]
})

prediction = model.predict(new_passenger)
probability = model.predict_proba(new_passenger)

print("Prediction:", prediction[0])
print("Survival probability:", probability[0][1])

## Decision weight

In [ ]:
import pandas as pd

feature_importance = pd.Series(model.feature_importances_, index=X.columns)
print(feature_importance.sort_values(ascending=False))